### wap to demonstarte the working of the decision tree based on ID3 algorithm.use an appprociate data set for building the decision tree and apply this knowledge to classify a new sample .

In [51]:
import pandas as pd
import numpy as np
import math

In [45]:
# Load the Play Tennis dataset
df = pd.read_csv('../exp5/tennisdata.csv')
print(df.head())
print("\nDataset shape:", df.shape)

    Outlook Temperature Humidity  Windy PlayTennis
0     Sunny         Hot     High  False         No
1     Sunny         Hot     High   True         No
2  Overcast         Hot     High  False        Yes
3     Rainy        Mild     High  False        Yes
4     Rainy        Cool   Normal  False        Yes

Dataset shape: (14, 5)


In [46]:
# Function to calculate entropy
def entropy(data, target_attr):
    values = data[target_attr].unique()
    entropy_val = 0
    for value in values:
        fraction = data[target_attr].value_counts()[value] / len(data[target_attr])
        entropy_val += -fraction * math.log2(fraction)
    return entropy_val

In [47]:
# Function to calculate information gain
def information_gain(data, attr, target_attr):
    total_entropy = entropy(data, target_attr)
    values = data[attr].unique()
    weighted_entropy = 0
    for value in values:
        subset = data[data[attr] == value]
        weighted_entropy += (len(subset) / len(data)) * entropy(subset, target_attr)
    return total_entropy - weighted_entropy

In [48]:
# ID3 Algorithm to build decision tree
def id3(data, original_data, features, target_attr, parent_node_class=None):
    # If all target values are the same, return that value
    if len(data[target_attr].unique()) <= 1:
        return data[target_attr].unique()[0]
    
    # If dataset is empty, return the majority class of parent node
    elif len(data) == 0:
        return parent_node_class
    
    # If no features left, return majority class
    elif len(features) == 0:
        return data[target_attr].value_counts().idxmax()
    
    else:
        # Set the default value for this node to majority class
        parent_node_class = data[target_attr].value_counts().idxmax()
        
        # Select the feature with the highest information gain
        item_values = [information_gain(data, feature, target_attr) for feature in features]
        best_feature_index = np.argmax(item_values)
        best_feature = features[best_feature_index]
        
        # Create the tree structure
        tree = {best_feature: {}}
        
        # Remove the best feature from the features list
        features = [i for i in features if i != best_feature]
        
        # Grow a branch for each possible value of the best feature
        for value in data[best_feature].unique():
            value = value
            sub_data = data[data[best_feature] == value]
            subtree = id3(sub_data, data, features, target_attr, parent_node_class)
            tree[best_feature][value] = subtree
        
        return tree

In [49]:
# Function to predict using the decision tree
def predict(query, tree, default='Yes'):
    for key in list(query.keys()):
        if key in list(tree.keys()):
            try:
                result = tree[key][query[key]]
            except:
                return default
            result = tree[key][query[key]]
            if isinstance(result, dict):
                return predict(query, result)
            else:
                return result

In [52]:
# Build the decision tree
features = list(df.columns[:-1])  # All columns except the target
target = df.columns[-1]  # Last column is the target

tree = id3(df, df, features, target)
print("Decision Tree:")
print(tree)

Decision Tree:
{'Outlook': {'Sunny': {'Humidity': {'High': 'No', 'Normal': 'Yes'}}, 'Overcast': 'Yes', 'Rainy': {'Windy': {np.False_: 'Yes', np.True_: 'No'}}}}


In [53]:
# Classify a new sample
new_sample = {
    'Outlook': 'Sunny',
    'Temperature': 'Cool',
    'Humidity': 'High',
    'Windy': 'True'
}

prediction = predict(new_sample, tree)
print(f"\nPrediction for new sample {new_sample}: {prediction}")

# Test with another sample
another_sample = {
    'Outlook': 'Overcast',
    'Temperature': 'Hot',
    'Humidity': 'Normal',
    'Windy': 'False'
}

prediction2 = predict(another_sample, tree)
print(f"\nPrediction for another sample {another_sample}: {prediction2}")


Prediction for new sample {'Outlook': 'Sunny', 'Temperature': 'Cool', 'Humidity': 'High', 'Windy': 'True'}: No

Prediction for another sample {'Outlook': 'Overcast', 'Temperature': 'Hot', 'Humidity': 'Normal', 'Windy': 'False'}: Yes


In [54]:
# Explanation:
# The decision tree shows that Outlook is the root node.
# - If Outlook is Sunny, check Humidity: High -> No, Normal -> Yes
# - If Outlook is Overcast, directly Yes
# - If Outlook is Rainy, check Windy: False -> Yes, True -> No

# The predictions match the expected behavior based on the dataset.